# CardioSentry: Grad-CAM Explainability on ConvNeXt-V2

This notebook applies **Grad-CAM (Gradient-weighted Class Activation Mapping)** to the **ConvNeXt-V2 Tiny** model trained on the CardioSentry 12-lead paper-ECG corpus.

### Supported Diagnoses:
1. **STEMI** — ST-Elevation Myocardial Infarction
2. **AF** — Atrial Fibrillation
3. **LVH** — Left Ventricular Hypertrophy
4. **NORMAL** — Normal ECG Rhythm

### Architecture & Geometry:
- **Backbone**: `ConvNeXt-V2-Tiny` with Global Response Normalization (`GRN`)
- **Resolution**: $768 \times 1024$ (native sheet aspect ratio)
- **Target Feature Map**: `stages.3` (or `stages.3.blocks.2`), yielding a $24 \times 32$ grid with 768 channels

In [ ]:
# 1. Install dependencies
!pip install -q timm torch torchvision opencv-python matplotlib pillow

In [ ]:
# 2. Optional: Mount Google Drive if your checkpoint is stored there
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_PATH = Path('/content/drive/MyDrive/CardioSentry/runs/convnextv2_tiny@nat768/weights/best.pt')
    print(f"Checking Drive checkpoint: {CKPT_PATH.exists()}")
except Exception:
    CKPT_PATH = Path('convnextv2_tiny@nat768.pt')
    print(f"Local checkpoint fallback: {CKPT_PATH.exists()}")

In [ ]:
# 3. Imports and device configuration
import cv2
import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms as T

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

LABELS = ["STEMI", "AF", "LVH", "NORMAL"]
DISPLAY_NAMES = {
    "STEMI": "ST-Elevation MI (STEMI)",
    "AF": "Atrial Fibrillation",
    "LVH": "Left Ventricular Hypertrophy",
    "NORMAL": "Normal Rhythm",
}
DEFAULT_THRESHOLDS = [0.73, 0.65, 0.92, 0.88]

In [ ]:
# 4. Load ConvNeXt-V2 Tiny model
def load_convnext(weights_file: Path | None = None):
    has_weights = weights_file is not None and weights_file.is_file()
    
    print(f"Building convnextv2_tiny.fcmae_ft_in1k (pretrained={not has_weights})...")
    model = timm.create_model(
        'convnextv2_tiny.fcmae_ft_in1k',
        pretrained=not has_weights,
        num_classes=4,
    )
    
    thresholds = list(DEFAULT_THRESHOLDS)
    
    if has_weights:
        print(f"Loading checkpoint: {weights_file}")
        blob = torch.load(weights_file, map_location='cpu', weights_only=False)
        state = blob['model'] if isinstance(blob, dict) and 'model' in blob else blob
        model.load_state_dict(state, strict=True)
        if isinstance(blob, dict) and 'thresholds' in blob:
            thresholds = [float(t) for t in blob['thresholds']]
        print(f"Loaded successfully. Trained thresholds: {thresholds}")
    else:
        print("Using timm ImageNet pretrained weights with default 0.5 thresholds.")
        thresholds = [0.5, 0.5, 0.5, 0.5]
        
    model.to(device)
    model.eval()
    return model, thresholds

model, thresholds = load_convnext(CKPT_PATH if CKPT_PATH.exists() else None)

In [ ]:
# 5. Preprocess ECG sheet
def preprocess_image(image_path_or_pil, input_hw=(768, 1024)):
    if isinstance(image_path_or_pil, (str, Path)):
        img = Image.open(image_path_or_pil).convert('RGB')
    else:
        img = image_path_or_pil.convert('RGB')
        
    h, w = input_hw
    resized = img.resize((w, h), Image.Resampling.BICUBIC)
    rgb_np = np.array(resized, dtype=np.uint8)
    
    tf = T.Compose([
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    tensor = tf(resized).unsqueeze(0)
    return tensor, rgb_np

In [ ]:
# 6. Grad-CAM Engine
def compute_gradcam(model, input_tensor, target_layer_name='stages.3'):
    named_mods = dict(model.named_modules())
    target_mod = named_mods[target_layer_name]
    
    activation = {}
    def hook(_m, _inp, out):
        t = out[0] if isinstance(out, (tuple, list)) else out
        if isinstance(t, torch.Tensor) and t.dim() == 4:
            leaf = t.detach().requires_grad_(True)
            activation['act'] = leaf
            torch.set_grad_enabled(True)
            return leaf
        return None
        
    handle = target_mod.register_forward_hook(hook)
    try:
        with torch.no_grad():
            x = input_tensor.to(device)
            logits = model(x)
            if logits.dim() == 1:
                logits = logits.unsqueeze(0)
            probs = torch.sigmoid(logits)[0]
            
            acts = activation['act'] # (1, C, H, W)
            cams = {}
            
            for idx, label in enumerate(LABELS):
                score = logits[0, idx]
                grads = torch.autograd.grad(score, acts, retain_graph=(idx < len(LABELS) - 1))[0]
                alpha = grads.mean(dim=(2, 3), keepdim=True)
                cam = torch.relu((alpha * acts).sum(dim=1, keepdim=True))
                cam = cam.squeeze().detach().cpu().numpy().astype(np.float32)
                
                c_min, c_max = cam.min(), cam.max()
                if c_max - c_min > 1e-8:
                    cam = (cam - c_min) / (c_max - c_min)
                else:
                    cam = np.zeros_like(cam)
                cams[label] = cam
    finally:
        handle.remove()
        
    return probs.cpu(), cams, acts.shape

In [ ]:
# 7. Overlay and Plotting
def overlay_cam(rgb_img, cam_2d, alpha=0.45, colormap=cv2.COLORMAP_TURBO):
    h, w = rgb_img.shape[:2]
    cam_resized = cv2.resize(cam_2d, (w, h), interpolation=cv2.INTER_LINEAR)
    heat_uint8 = np.uint8(255 * cam_resized)
    heat_color = cv2.applyColorMap(heat_uint8, colormap)
    heat_color = cv2.cvtColor(heat_color, cv2.COLOR_BGR2RGB)
    blended = np.float32(heat_color) * alpha + np.float32(rgb_img) * (1.0 - alpha)
    return np.uint8(np.clip(blended, 0, 255))

def visualize_ecg_gradcam(rgb_img, probs, cams, thresholds, grid_shape):
    fig, axes = plt.subplots(5, 1, figsize=(14, 20), dpi=120)
    fig.patch.set_facecolor('#121417')
    
    axes[0].imshow(rgb_img)
    axes[0].set_title(f"Input ECG Sheet ({rgb_img.shape[1]}x{rgb_img.shape[0]} px) | Feature Grid: {grid_shape[3]}x{grid_shape[2]} | Channels: {grid_shape[1]}",
                      color='white', fontsize=12, fontweight='bold', pad=6)
    axes[0].axis('off')
    
    for i, label in enumerate(LABELS):
        ax = axes[i + 1]
        p = float(probs[i])
        th = float(thresholds[i])
        is_pos = p >= th
        
        overlay = overlay_cam(rgb_img, cams[label])
        ax.imshow(overlay)
        
        color = '#FF4444' if (is_pos and label != 'NORMAL') else ('#00E676' if is_pos else '#AAAAAA')
        status = 'POSITIVE' if is_pos else 'NEGATIVE'
        ax.set_title(f"{DISPLAY_NAMES[label]} | Score: {p:.4f} vs Thresh: {th:.2f} [{status}]",
                     color=color, fontsize=12, fontweight='bold', pad=6)
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()

In [ ]:
# 8. Run demo on a sample ECG sheet
# Replace with your own image path or upload via files.upload()
test_image_path = Path('cardiosentry-application/tests/fixtures/corpus_sheet.jpg')

if not test_image_path.exists():
    print("Creating synthetic blank ECG canvas for demonstration...")
    dummy_img = Image.new('RGB', (1024, 768), color=(250, 245, 240))
    tensor, rgb = preprocess_image(dummy_img)
else:
    tensor, rgb = preprocess_image(test_image_path)

probs, cams, grid_shape = compute_gradcam(model, tensor, target_layer_name='stages.3')
visualize_ecg_gradcam(rgb, probs, cams, thresholds, grid_shape)